# ML Pipeline - Phase 2: Huấn luyện Mô hình, Hyperparameter Tuning & Đánh giá Y tế (PTB-XL)

Thực hiện đầy đủ:
1. **Cân Bằng Trọng Số Lớp (Class Weight Balancing)**: Sử dụng `class_weight='balanced'` & `scale_pos_weight=3.8` để tối ưu hài hòa giữa Accuracy (~75-80%) và Recall (~55-65%).
2. **Phân loại Nhị phân (Binary ML)**: Fine-tune **Logistic Regression** & **XGBoost**.
3. **Deep Learning (Neural Network / MLP)**: Tuning Hyperparameters `hidden_layer_sizes=(64, 32)`.
4. **Tổ hợp Mô hình (Soft Voting & Stacking Ensemble)**.
5. **Đánh giá Y tế Chuyên sâu**: **Accuracy**, **Recall**, **F1-Score**, **ROC-AUC**, **Specificity**.

In [ ]:
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb, lightgbm as lgb, joblib
print('✅ Nạp thành công thư viện Training cho PTB-XL!')

### 1. Nạp Dữ Liệu PTB-XL & Tỉ Lệ Lớp

In [ ]:
minmax_fname = 'ptbxl_minmax_scaled.csv'
zscore_fname = 'ptbxl_zscore_scaled.csv'
data_dir_candidates = ['../../../data/processed', '../../data/processed', 'data/processed', '../../../data/features', '../../data/features', 'data/features']
data_dir = next((d for d in data_dir_candidates if os.path.exists(os.path.join(d, minmax_fname))), '../../data/processed')

df_minmax = pd.read_csv(os.path.join(data_dir, minmax_fname))
df_zscore = pd.read_csv(os.path.join(data_dir, zscore_fname))

X_mm = df_minmax.drop(columns=['status'])
y_mm = df_minmax['status']
X_zs = df_zscore.drop(columns=['status'])
y_zs = df_zscore['status']

neg_count, pos_count = np.bincount(y_mm)
pos_scale = neg_count / pos_count
print(f"📊 Tỉ lệ nhãn PTB-XL: Bình thường={neg_count} ({neg_count/len(y_mm):.1%}) | Bất thường={pos_count} ({pos_count/len(y_mm):.1%})")
print(f"⚖️ Hệ số cân bằng class_weight/scale_pos_weight = {pos_scale:.2f}")

Xmm_train, Xmm_test, y_train, y_test = train_test_split(X_mm, y_mm, test_size=0.2, random_state=42, stratify=y_mm)
Xzs_train, Xzs_test, _, _ = train_test_split(X_zs, y_zs, test_size=0.2, random_state=42, stratify=y_zs)
print(f'Tập Train PTB-XL: {len(y_train)} mẫu | Tập Test: {len(y_test)} mẫu')

### 2. Fine-Tuning Mô Hình CÓ CÂN BẰNG TRỌNG SỐ LỚP (PTB-XL)

In [ ]:
# 1. Logistic Regression với class_weight='balanced'
grid_lr = GridSearchCV(LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'), 
                       {'C': [0.01, 0.1, 1.0, 10.0], 'penalty': ['l2'], 'solver': ['lbfgs', 'liblinear']}, 
                       cv=3, n_jobs=-1, scoring='f1')
grid_lr.fit(Xzs_train, y_train)
best_lr = grid_lr.best_estimator_

# 2. XGBoost với scale_pos_weight
grid_xgb = GridSearchCV(xgb.XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=pos_scale), 
                        {'max_depth': [3, 4, 5], 'learning_rate': [0.01, 0.05, 0.1], 'n_estimators': [100, 200]}, 
                        cv=3, n_jobs=-1, scoring='f1')
grid_xgb.fit(Xmm_train, y_train)
best_xgb = grid_xgb.best_estimator_

# 3. Deep Learning MLP
grid_mlp = GridSearchCV(MLPClassifier(solver='adam', activation='logistic', random_state=42, max_iter=200, early_stopping=False), 
                        {'hidden_layer_sizes': [(64, 32), (32, 16, 8), (64, 32, 16)], 'learning_rate_init': [0.001, 0.01]}, 
                        cv=3, n_jobs=-1, scoring='f1')
grid_mlp.fit(Xmm_train, y_train)
best_mlp = grid_mlp.best_estimator_

print('✅ Hoàn tất Fine-tuning Logistic Regression, XGBoost & Deep Learning MLP!')

### 3. Huấn luyện Ensemble Learning

In [ ]:
svm_model = SVC(probability=True, class_weight='balanced', random_state=42).fit(Xzs_train, y_train)
rf_model = RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42).fit(Xmm_train, y_train)
et_model = ExtraTreesClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42).fit(Xmm_train, y_train)
lgb_model = lgb.LGBMClassifier(random_state=42, verbose=-1, scale_pos_weight=pos_scale, n_estimators=200, max_depth=5).fit(Xmm_train, y_train)

voting_clf = VotingClassifier(
    estimators=[('svm', svm_model), ('xgb', best_xgb), ('rf', rf_model), ('lr', best_lr)],
    voting='soft', weights=[1.5, 2.0, 1.0, 1.5]
).fit(Xmm_train, y_train)

stacking_clf = StackingClassifier(
    estimators=[('xgb', best_xgb), ('mlp', best_mlp), ('svm', svm_model), ('et', et_model), ('rf', rf_model)],
    final_estimator=LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    cv=5,
    n_jobs=-1
).fit(Xmm_train, y_train)

print('✅ Huấn luyện thành công Soft Voting & Stacking Ensemble!')

### 4. Bảng Đánh Giá Tổng Hợp Thước Đo Y Tế (Ngưỡng 0.5 Chuẩn Cân Bằng)

In [ ]:
all_models = {
    'Logistic Regression (Balanced)': (best_lr, Xzs_test),
    'Random Forest (Balanced)': (rf_model, Xmm_test),
    'LGBM Classifier (Weighted)': (lgb_model, Xmm_test),
    'XGBoost (Weighted)': (best_xgb, Xmm_test),
    'Neural Network / MLP': (best_mlp, Xmm_test),
    'Soft Voting Ensemble': (voting_clf, Xmm_test),
    'Stacking Ensemble': (stacking_clf, Xmm_test)
}

eval_results = []
roc_data = {}
conf_matrices = {}

# Dùng ngưỡng chuẩn 0.5 khi đã áp dụng class_weight/scale_pos_weight
THRESHOLD = 0.50

for name, (model, X_test_set) in all_models.items():
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test_set)[:, 1]
        y_pred = (y_prob >= THRESHOLD).astype(int)
    else:
        y_pred = model.predict(X_test_set)
        y_prob = y_pred
        
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    roc_auc = roc_auc_score(y_test, y_prob)
    
    eval_results.append({
        'Model': name,
        'Accuracy': acc,
        'Recall (Sensitivity)': rec,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'Precision': prec,
        'Specificity': spec,
        'False Negative (FN)': fn
    })
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_data[name] = (fpr, tpr, roc_auc)
    conf_matrices[name] = cm

df_eval = pd.DataFrame(eval_results).sort_values(by=['F1-Score', 'Accuracy'], ascending=False).reset_index(drop=True)
badges = ['🥇 ', '🥈 ', '🥉 '] + [''] * (len(df_eval) - 3)
df_eval['Rank'] = [f'{badges[i]}#{i+1}' for i in range(len(df_eval))]
df_eval = df_eval[['Rank', 'Model', 'Accuracy', 'Recall (Sensitivity)', 'F1-Score', 'ROC-AUC', 'Precision', 'Specificity', 'False Negative (FN)']]

styled_df = df_eval.style.format({
    'Accuracy': '{:.2%}', 'Recall (Sensitivity)': '{:.2%}', 'F1-Score': '{:.2%}', 'ROC-AUC': '{:.4f}',
    'Precision': '{:.2%}', 'Specificity': '{:.2%}', 'False Negative (FN)': '{:d}'
}).background_gradient(cmap='Blues', subset=['Accuracy', 'Recall (Sensitivity)', 'F1-Score', 'ROC-AUC'])

from IPython.display import display, HTML
display(HTML("<h2 style='text-align: center; color: #0f172a;'>🏆 BẢNG XẾP HẠNG THƯỚC ĐO Y TẾ (CÂN BẰNG HÀI HÒA ACCURACY & RECALL)</h2>"))
display(styled_df)

### 5. Trực quan hóa Đường Cong ROC & Confusion Matrices (PTB-XL)

In [ ]:
plt.figure(figsize=(10, 8))
for name, (fpr, tpr, roc_auc) in roc_data.items():
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall / Sensitivity)')
plt.title('So sánh Đường Cong ROC Dữ Liệu PTB-XL (Balanced Weights)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

### 6. Lưu trữ Models Trained V2 (PTB-XL)

In [ ]:
models_dir_candidates = ['../../../models/ptbxl', '../../models/ptbxl', 'models/ptbxl']
models_dir = next((d for d in models_dir_candidates if os.path.exists(os.path.dirname(d))), '../../models/ptbxl')
os.makedirs(models_dir, exist_ok=True)
joblib.dump(best_lr, os.path.join(models_dir, 'logistic_regression.pkl'))
joblib.dump(best_xgb, os.path.join(models_dir, 'xgboost.pkl'))
joblib.dump(best_mlp, os.path.join(models_dir, 'neural_network_mlp.pkl'))
joblib.dump(voting_clf, os.path.join(models_dir, 'voting_ensemble.pkl'))
joblib.dump(stacking_clf, os.path.join(models_dir, 'stacking_ensemble.pkl'))
print(f'✅ Đã lưu thành công các mô hình vào: {models_dir}')